# CarDekho used-car price exploration

This notebook reads the original eight-column CSV and checks its quality before
summarizing prices. See the [dataset guide](../data/README.md) for the source,
license, currency evidence, and download steps. All prices are reported in
Indian rupees. Run the cells from top to bottom with the project environment.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from ml_benchmark.datasets import load_csv, validate_dataset

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file())
raw_path = root / "data/raw/CAR DETAILS FROM CAR DEKHO.csv"
output = root / "results/published"
columns = ["name", "year", "selling_price", "km_driven", "fuel",
           "seller_type", "transmission", "owner"]
cars = load_csv(raw_path, required_columns=columns)

## 1. Check data quality

We count a row as an exact repeat when all eight fields match an earlier row.
A blank value is counted separately from a value that fails a rule. The checks
below look for clear format and range problems. A suspicious value can pass
these checks, so we also inspect unusually low odometer readings.

In [ ]:
category_values = {
    "fuel": ["Petrol", "Diesel", "CNG", "LPG", "Electric"],
    "seller_type": ["Individual", "Dealer", "Trustmark Dealer"],
    "transmission": ["Manual", "Automatic"],
    "owner": ["First Owner", "Second Owner", "Third Owner",
              "Fourth & Above Owner", "Test Drive Car"],
}

def whole_number_in_range(values, minimum, maximum):
    numbers = pd.to_numeric(values, errors="coerce")
    return numbers.between(minimum, maximum) & numbers.mod(1).eq(0)

rules = {
    "name": lambda values: values.astype(str).str.strip().ne(""),
    "year": lambda values: whole_number_in_range(values, 1886, 2023),
    "selling_price": lambda values: whole_number_in_range(values, 1, float("inf")),
    "km_driven": lambda values: whole_number_in_range(values, 0, float("inf")),
}
rules.update({
    column: (lambda values, allowed=allowed: values.isin(allowed))
    for column, allowed in category_values.items()
})

quality = validate_dataset(cars, rules=rules)
print(f"{quality.rows:,} rows, {quality.columns} columns")
print(f"{quality.duplicate_rows:,} exact repeat rows")
print(quality.columns_report.to_string())
print(f"Rows below 100 km: {cars['km_driven'].lt(100).sum()}")
print(f"Minimum distance: {cars['km_driven'].min():,} km")

## 2. Look at the price distribution

Use the raw rows for the main analysis. The CSV has no listing ID, so exact
matches could include separate listings with the same recorded details. Compare
the mean and median after dropping exact matches as a sensitivity check. One
lakh equals 100,000 rupees. The histogram includes the full price range.

In [ ]:
price = cars["selling_price"]
print(price.describe().round(2).to_string())
print(f"Listings at or below INR 10 lakh: {price.le(1_000_000).mean():.1%}")
print(cars.drop_duplicates()["selling_price"].agg(["size", "mean", "median"]).round(2))

source = "Source: Nehal Birla et al., CarDekho Vehicle dataset, Kaggle v4 (ODbL / DbCL)"

def publish(fig, filename):
    fig.text(0.01, 0.01, source, fontsize=8, color="#495766")
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(output / filename, dpi=160, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

output.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(price / 100_000, bins=45, color="#246b83")
ax.axvline(price.median() / 100_000, color="#9d421f", linestyle="--", label="Median")
ax.set(title="Most listings cluster at the lower end of the price range",
       xlabel="Listed selling price (INR lakh)", ylabel="Listings")
ax.legend(frameon=False)
publish(fig, "cardekho-price-distribution.png")

## 3. Compare price with vehicle year and distance driven

Spearman correlation measures whether two values tend to move in the same
direction. It ranges from -1 to +1. The charts show group medians with listing
counts so readers can see which estimates come from smaller groups.

In [ ]:
def group_prices(groups):
    return cars.groupby(groups, observed=True, sort=True)["selling_price"].agg(
        listings="size", median_inr="median"
    )

numeric = cars[["year", "km_driven", "selling_price"]]
ranks = numeric.rank()
print(ranks.corr()["selling_price"].drop("selling_price").round(3).to_string())

years = pd.cut(cars["year"], [1991, 2000, 2005, 2010, 2015, 2020, 2023],
               labels=["1992–00", "2001–05", "2006–10", "2011–15", "2016–20", "2021–23"])
distance = pd.cut(cars["km_driven"], [-1, 30_000, 60_000, 90_000, 120_000, float("inf")],
                  labels=["0–30k", ">30–60k", ">60–90k", ">90–120k", ">120k"])

def median_bars(ax, summary, title):
    labels = [f"{label}\n(n={count:,})" for label, count in summary["listings"].items()]
    ax.bar(labels, summary["median_inr"] / 100_000, color="#246b83")
    ax.set(title=title, ylabel="Median price (INR lakh)")
    ax.tick_params(axis="x", labelsize=8)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
median_bars(axes[0], group_prices(years), "Vehicle year")
median_bars(axes[1], group_prices(distance), "Kilometres driven")
publish(fig, "cardekho-price-year-distance.png")

## 4. Compare price across categories

Check the listing counts beside each median. A difference can reflect a mix of
vehicle ages, models, and mileage. For vehicle name, we report the five most
frequent names to keep the comparison readable.

In [ ]:
tables = {column: group_prices(cars[column]) for column in category_values}
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for axis, (column, summary) in zip(axes.flat, tables.items(), strict=True):
    print(summary.to_string(), end="\n\n")
    labels = [f"{label} (n={count:,})" for label, count in summary["listings"].items()]
    axis.barh(labels, summary["median_inr"] / 100_000, color="#246b83")
    axis.set(title=column.replace("_", " ").capitalize(), xlabel="Median price (INR lakh)")
names = cars["name"].value_counts().head(5).index
print(group_prices(cars["name"]).loc[names].to_string())
publish(fig, "cardekho-price-categories.png")

## 5. Read the results

The right tail explains why the average price is above the median. Group
medians describe these records and do not isolate the effect of a single
feature. This file has no listing IDs, listing dates, or transaction
confirmation. Review the repeated rows and unusually low odometer reading
before deciding how to split or clean the data for a benchmark.